In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F


In [2]:
import numpy as np
import pandas as pd


## Input text definition and tokenizer 

In [39]:
import torch
from torch.utils.data import Dataset

class YearAnnotatedDataset(Dataset):
    def __init__(self, sequences, main_vocab, year_vocab):
        """
        sequences: list of tokenized sequences (e.g., [["YEAR_0", "born", "YEAR_1", "flu", ...]])
        main_vocab: dict mapping word tokens -> int
        year_vocab: dict mapping "YEAR_n" tokens -> int (usually 0–99)
        """
        self.sequences = sequences
        self.main_vocab = main_vocab
        self.year_vocab = year_vocab

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        tokens = self.sequences[idx]
        input_ids = []
        year_ids = []
        current_year = None

        for token in tokens:
            if token in self.year_vocab:
                current_year = self.year_vocab[token]
            elif token in self.main_vocab:
                input_ids.append(self.main_vocab[token])
                year_ids.append(current_year)
            else:
                raise ValueError(f"Unknown token: {token}")

        return torch.tensor(input_ids, dtype=torch.long), torch.tensor(year_ids, dtype=torch.long)


In [61]:
def collate_batch(batch, pad_id=0, pad_year=-1):
    """
    Pads input_ids and year_ids to the same max length in the batch.
    """
    input_ids, year_ids = zip(*batch)
    max_len = max(seq.size(0) for seq in input_ids) # len(seq)

    padded_inputs = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
    padded_years  = torch.full((len(batch), max_len), pad_year, dtype=torch.long)

    for i, (input_seq, year_seq) in enumerate(zip(input_ids, year_ids)):
        length = input_seq.size(0)
        padded_inputs[i, :length] = input_seq
        padded_years[i, :length] = year_seq

    return padded_inputs, padded_years

# """
#     Pads a batch of (input_ids, year_ids) to the same sequence length.

#     Args:
#         batch: List of (input_ids_tensor, year_ids_tensor)
#         pad_token_id: ID used to pad input tokens
#         pad_year_id: ID used to pad year embeddings (should be ignored by year embedding layer)

#     Returns:
#         Tuple of:
#             - padded_input_ids: (B, L)
#             - padded_year_ids:  (B, L)
#     """
#  for list input 
#  def collate_life_event_batch(batch, pad_token_id=0, pad_year_id=-1):
#     input_ids_list, year_ids_list = zip(*batch)
#     max_len = max(len(seq) for seq in input_ids_list)
#     batch_size = len(batch)

#     padded_input_ids = torch.full((batch_size, max_len), pad_token_id, dtype=torch.long)
#     padded_year_ids = torch.full((batch_size, max_len), pad_year_id, dtype=torch.long)

#     for i in range(batch_size):
#         seq_len = input_ids_list[i].size(0)
#         padded_input_ids[i, :seq_len] = input_ids_list[i]
#         padded_year_ids[i, :seq_len] = year_ids_list[i]
# return padded_input_ids, padded_year_ids

In [41]:
from torch.utils.data import DataLoader

# Example vocab
main_vocab = {'born': 0, 'had': 1, 'flu': 2, 'got': 3, 'cancer': 4, 'died': 5}
year_vocab = {f'YEAR_{i}': i for i in range(100)}

# Tokenized sequences
data = [
    ["YEAR_0", "born", "YEAR_1", "had", "flu", "YEAR_65", "got", "cancer", "YEAR_99", "died"],
    ["YEAR_0", "born", "YEAR_20", "had", "flu", "YEAR_21", "died"]
]

dataset = YearAnnotatedDataset(data, main_vocab, year_vocab)
loader = DataLoader(dataset, batch_size=2, collate_fn=collate_batch)

for input_ids, year_ids in loader:
    print("Input IDs:\n", input_ids)
    print("Year IDs:\n", year_ids)

# Padding token for missing words is 0 (can be changed).
# Padding token for missing years is -1.
# You can mask these during loss computation if needed.



Input IDs:
 tensor([[0, 1, 2, 3, 4, 5],
        [0, 1, 2, 5, 0, 0]])
Year IDs:
 tensor([[ 0,  1,  1, 65, 65, 99],
        [ 0, 20, 20, 21, -1, -1]])


## ✅  Transformer Decoder "YearAwareTransformer"
We use the TransformerDecoder which is a decoder-only model, like GPT-style transformers, since we're modeling a sequence where each token generation only depends on the previous ones.

- Word Embedding (word_embedding): Embeds words from the main vocabulary.

- Position Embedding (pos_embedding): Adds positional information to the word embeddings, to keep track of the token order.

- Year Embedding (year_embedding): We add year embeddings for tokens that represent years.

- Masking: The generate_square_subsequent_mask is used to prevent attending to future positions (for causal language modeling).


In [48]:
import torch
import torch.nn as nn

class YearAwareTransformer(nn.Module):
    def __init__(self, vocab_size, num_years, d_model=128, nhead=4, num_layers=2, max_seq_len=128):
        super().__init__()
        self.d_model = d_model

        # Word and position embeddings
        self.word_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_seq_len, d_model)

        # Year embeddings (separate)
        self.year_embedding = nn.Embedding(num_years, d_model)

        # Transformer Decoder (GPT-style, decoder-only model)
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)

        self.output_head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids, year_ids):
        """
        input_ids: [B, L] - token indices from the main vocab
        year_ids:  [B, L] - indices into year vocab or -1 if not a year token
        """
        batch_size, seq_len = input_ids.size()
        pos_ids = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)

        # Word and position embeddings
        word_emb = self.word_embedding(input_ids)  # [B, L, D]
        pos_emb = self.pos_embedding(pos_ids)      # [B, L, D]

        # Handle year embedding
        year_emb = torch.zeros_like(word_emb)
        for i in range(batch_size):
            for j in range(seq_len):
                if year_ids[i, j] != -1:
                    year_emb[i, j] = self.year_embedding(year_ids[i, j])  # Use year embedding only if year is valid

        # Final embedding = word + position + year
        x = word_emb + pos_emb + year_emb
        x = x.transpose(0, 1)  # Transformer expects [L, B, D]

        # Generate the causal mask (look-ahead mask)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(input_ids.device)

        # Pass through the transformer decoder
        output = self.transformer_decoder(x, x, tgt_mask=tgt_mask)

        output = output.transpose(0, 1)  # [B, L, D]
        logits = self.output_head(output)  # [B, L, vocab_size]
        return logits



  *Example forward pass and loss calculation:*

### 1. Training Setup:


In [53]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.optim as optim

# Define the vocabulary
main_vocab = {'born': 0, 'had': 1, 'flu': 2, 'got': 3, 'cancer': 4, 'died': 5}
year_vocab = {f'YEAR_{i}': i for i in range(100)}  # YEAR_0 to YEAR_99

# Tokenized sequences (example data)
data = [
    ["YEAR_0", "born", "YEAR_1", "had", "flu", "YEAR_65", "got", "cancer", "YEAR_99", "died"],
    ["YEAR_0", "born", "YEAR_20", "had", "flu", "YEAR_21", "died"]
]

# Initialize Dataset and DataLoader
dataset = YearAnnotatedDataset(data, main_vocab, year_vocab)
loader = DataLoader(dataset, batch_size=2, collate_fn=collate_batch)

# Initialize the model
model = YearAwareTransformer(vocab_size=len(main_vocab), num_years=100, d_model=128, nhead=4, num_layers=2)

# Define optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Define loss function
criterion = nn.CrossEntropyLoss()


### 2. Training Loop:


In [55]:
num_epochs = 10  # Set the number of epochs

for epoch in range(num_epochs):
    model.train()  # Set model to training mode
    total_loss = 0

    for input_ids, year_ids in loader:
        optimizer.zero_grad()

        # Forward pass: get the model output (logits)
        logits = model(input_ids, year_ids)

        # Calculate the loss using the original input_ids (as labels)
        # CrossEntropyLoss expects the target to be in the shape [B, L], i.e., input_ids without padding
        loss = criterion(logits.view(-1, len(main_vocab)), input_ids.view(-1))

        # Backward pass: compute gradients
        loss.backward()

        # Optimizer step: update weights
        optimizer.step()

        # Track total loss
        total_loss += loss.item()

    # Print the average loss for the epoch
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss / len(loader):.4f}")


Epoch 1/10, Loss: 1.7230
Epoch 2/10, Loss: 0.9526
Epoch 3/10, Loss: 0.3040
Epoch 4/10, Loss: 0.1008
Epoch 5/10, Loss: 0.0599
Epoch 6/10, Loss: 0.0450
Epoch 7/10, Loss: 0.0316
Epoch 8/10, Loss: 0.0221
Epoch 9/10, Loss: 0.0174
Epoch 10/10, Loss: 0.0122


### 3. Evaluation and Generation:


In [127]:
def generate_sequence(model, start_tokens, year_ids, max_len=5):
    model.eval()  # Set model to evaluation mode
    generated_tokens = start_tokens.copy()

    # For each new token, predict the next token
    for _ in range(max_len):
        # Prepare input sequence for the model
        input_ids = torch.tensor([generated_tokens], dtype=torch.long)
        year_ids_tensor = torch.tensor([year_ids], dtype=torch.long)

        # Get the model's output logits
        logits = model(input_ids, year_ids_tensor)
        
        # Get the predicted token (greedy decoding, choose token with max probability)
        next_token_id = logits.argmax(dim=-1)[:, -1].item()  # [B, L, V], get last token's prediction
        generated_tokens.append(next_token_id)
        
        # Update year_ids (for simplicity, we can assume the same year as the previous token)
        year_ids.append(year_ids[-1])  # Just keep the last year's ID for simplicity
    
        if next_token_id == main_vocab['died']:  # Stop when 'died' token is generated
            break

    return generated_tokens


In [57]:
# Generate a sequence starting with 'YEAR_0' and 'born'
start_tokens = [year_vocab['YEAR_0'], main_vocab['born']]
year_ids = [year_vocab['YEAR_0'], year_vocab['YEAR_0']]

generated_ids = generate_sequence(model, start_tokens, year_ids)
generated_sequence = [list(main_vocab.keys())[list(main_vocab.values()).index(id)] for id in generated_ids]
print("Generated Sequence:", " ".join(generated_sequence))

Generated Sequence: Born Born Born flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu flu Born Born Born Born Born




### Taking data from long format csv (ID / age / event) into data for transformer
We need to:
- Group by person ID.
- Sort each group by age.
- Interleave year tokens (YEAR_#) and event tokens as a flat sequence for each person.
- Convert sequences into token IDs using main_vocab and year_vocab.
- Wrap this into a Dataset class.

The result will be:
- Each row in the CSV becomes a pair of YEAR_{age} and event.
- For each person, their timeline is built by sorting their events by age.
- Resulting token sequence looks like: ["YEAR_0", "born", "YEAR_1", "flu", "YEAR_65", "cancer", "YEAR_99", "died"].



In [85]:
import pandas as pd
from torch.utils.data import Dataset
import torch

class LifeEventDataset(Dataset):
    def __init__(self, df, main_vocab, year_vocab):
        self.main_vocab = main_vocab
        self.year_vocab = year_vocab

        # Sanity check
        assert {'patid', 'age', 'event'}.issubset(df.columns), "CSV must contain patid, age, event"

        # Group by person
        self.sequences = []
        grouped = df.groupby('patid')

        for _, group in grouped:
            group = group.sort_values('age')
            tokens = []
            for _, row in group.iterrows():
                year_token = f"YEAR_{int(row['age'])}"
                event_token = row['event']
                if year_token in year_vocab and event_token in main_vocab:
                    tokens.extend([year_token, event_token])
                else:
                    raise ValueError(f"Unknown year/event token: {year_token}, {event_token}")
            self.sequences.append(tokens)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        tokens = self.sequences[idx]
        input_ids = []
        year_ids = []
        current_year = None

        for token in tokens:
            if token in self.year_vocab:
                current_year = self.year_vocab[token]
            elif token in self.main_vocab:
                input_ids.append(self.main_vocab[token])
                year_ids.append(current_year)
            else:
                raise ValueError(f"Unknown token: {token}")

        return torch.tensor(input_ids, dtype=torch.long), torch.tensor(year_ids, dtype=torch.long)

## EXAMPLE 

In [206]:
# read data, create main vocab from all events in the data, create year_vocab from 0 to 110

df = pd.read_csv("Event_data.csv")
main_vocab = {word: index for index, word in enumerate(set(df.event))}
year_vocab = {f'YEAR_{i}': i for i in range(110)}
main_vocab


{'flu': 0, 't2dm': 1, 'cancer': 2, 'born': 3, 'died': 4}

In [207]:
# use function above to create a list input_ids and year_ids per patient 
dataset = LifeEventDataset(df, main_vocab, year_vocab)


In [213]:
# check what it gives for the first patient
input_ids, year_ids = dataset[0]
print(input_ids)  # tensor of token IDs
print(year_ids)   # tensor of year IDs aligned with each token

tensor([3, 0, 4])
tensor([ 0, 26, 82])


In [212]:
df[df['patid'] == 2]  # ok, age 0 26 and 82, events are 3 0 and 4 from above 

,patid,age,event
3,2,0,born
4,2,26,flu
5,2,57,cancer
6,2,74,died


In [203]:
del(model1)

In [204]:
loader = DataLoader(dataset, batch_size=2, collate_fn=collate_batch)

# Initialize the model
model1 = YearAwareTransformer(vocab_size=len(main_vocab), num_years=100, d_model=128, nhead=4, num_layers=2)

# Define optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Define loss function
criterion = nn.CrossEntropyLoss()


In [205]:
num_epochs = 200  # Set the number of epochs

for epoch in range(num_epochs):
    model1.train()  # Set model to training mode
    total_loss = 0

    for input_ids, year_ids in loader:
        optimizer.zero_grad()

        # Forward pass: get the model output (logits)
        logits = model1(input_ids, year_ids)

        # Calculate the loss using the original input_ids (as labels)
        # CrossEntropyLoss expects the target to be in the shape [B, L], i.e., input_ids without padding
        loss = criterion(logits.view(-1, len(main_vocab)), input_ids.view(-1))

        # Backward pass: compute gradients
        loss.backward()

        # Optimizer step: update weights
        optimizer.step()

        # Track total loss
        total_loss += loss.item()

    # Print the average loss for the epoch
    if epoch%5 ==0:
        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss / len(loader):.4f}")

Epoch 1/200, Loss: 1.8580
Epoch 6/200, Loss: 1.8452
Epoch 11/200, Loss: 1.8940
Epoch 16/200, Loss: 1.8940
Epoch 21/200, Loss: 1.8628
Epoch 26/200, Loss: 1.8768
Epoch 31/200, Loss: 1.9013
Epoch 36/200, Loss: 1.8872
Epoch 41/200, Loss: 1.8694
Epoch 46/200, Loss: 1.8886
Epoch 51/200, Loss: 1.9224
Epoch 56/200, Loss: 1.8835
Epoch 61/200, Loss: 1.8343
Epoch 66/200, Loss: 1.8848
Epoch 71/200, Loss: 1.8709
Epoch 76/200, Loss: 1.8839
Epoch 81/200, Loss: 1.8803
Epoch 86/200, Loss: 1.8883
Epoch 91/200, Loss: 1.8769
Epoch 96/200, Loss: 1.8573
Epoch 101/200, Loss: 1.8797
Epoch 106/200, Loss: 1.9011
Epoch 111/200, Loss: 1.8586
Epoch 116/200, Loss: 1.8638
Epoch 121/200, Loss: 1.8024
Epoch 126/200, Loss: 1.8906
Epoch 131/200, Loss: 1.8857
Epoch 136/200, Loss: 1.8752
Epoch 141/200, Loss: 1.8837
Epoch 146/200, Loss: 1.8877
Epoch 151/200, Loss: 1.8707
Epoch 156/200, Loss: 1.8935
Epoch 161/200, Loss: 1.8895
Epoch 166/200, Loss: 1.8476
Epoch 171/200, Loss: 1.8651
Epoch 176/200, Loss: 1.8863
Epoch 181/200,

In [188]:
# Generate a sequence starting with 'YEAR_0' and 'born'
start_tokens = [main_vocab['born']]
year_ids = [year_vocab['YEAR_0']]

generated_ids = generate_sequence(model1, start_tokens, year_ids, 20)
generated_sequence = [list(main_vocab.keys())[list(main_vocab.values()).index(id)] for id in generated_ids]
generated_years = [list(year_vocab.keys())[list(year_vocab.values()).index(id)] for id in generated_ids]
print("Generated Sequence:", " ".join(generated_sequence))

Generated Sequence: born flu t2dm born flu t2dm t2dm born born born t2dm born t2dm born born t2dm born born born born born


In [182]:
# why does it not end with die? 
# how to convert logits into probabilities over events? 
# how to predict years?? 
# WE NEED TO RE-DEFINE LOSS = Mispredict of the category + Mispredict in years 


In [190]:
logits

tensor([[[ 0.4567,  0.5527, -0.6791,  0.9534,  0.0422],
         [ 0.3966,  0.4477, -0.2782,  0.2806, -0.7163],
         [ 0.4368,  0.2411, -1.4035,  0.6794, -0.6336],
         [ 0.7454,  0.3384, -1.0065,  0.4935, -0.6050],
         [ 0.8396,  0.7808, -1.5167,  0.7223, -0.1414],
         [ 0.1565,  0.7804, -1.4710,  0.3198, -0.6380]]],
       grad_fn=<ViewBackward0>)

In [183]:
max_len = 5
model.eval()  # Set model to evaluation mode
generated_tokens = start_tokens.copy()

# For each new token, predict the next token
for _ in range(max_len):
    # Prepare input sequence for the model
    input_ids = torch.tensor([generated_tokens], dtype=torch.long)
    year_ids_tensor = torch.tensor([year_ids], dtype=torch.long)

    # Get the model's output logits
    logits = model1(input_ids, year_ids_tensor)
    print(logits) 
    
    # Get the predicted token (greedy decoding, choose token with max probability)
    next_token_id = logits.argmax(dim=-1)[:, -1].item()  # [B, L, V], get last token's prediction
    next_year_id = logits.argmax(dim=-1)[:, -1].item()  # [B, L, V], get last token's prediction

    print(list(main_vocab.keys())[next_token_id])
    
    generated_tokens.append(next_token_id)
        
    # Update year_ids (for simplicity, we can assume the same year as the previous token) NO !
    year_ids.append(year_ids[-1])  # Just keep the last year's ID for simplicity   -WE DONT WANT THIS :)
    
        
    if next_token_id == main_vocab['died']:  # Stop when 'died' token is generated
        break


tensor([[[ 0.1630,  0.4171, -0.6202,  0.7179,  0.0160],
         [-0.1120,  0.1468, -0.3578, -0.2616, -0.7972]]],
       grad_fn=<ViewBackward0>)
t2dm
tensor([[[ 0.2136,  0.5157, -0.6034,  0.6382, -0.1137],
         [ 0.1397,  0.3049, -0.1427, -0.1236, -0.8601],
         [ 0.2481, -0.0219, -1.5166,  0.3230, -0.7973]]],
       grad_fn=<ViewBackward0>)
born
tensor([[[ 0.2920,  0.4263, -0.8277,  0.8750, -0.0280],
         [ 0.1400,  0.2678, -0.3307,  0.0588, -0.8436],
         [ 0.2191, -0.0514, -1.6610,  0.4994, -0.7287],
         [ 0.5695,  0.1302, -1.0917,  0.4170, -0.5883]]],
       grad_fn=<ViewBackward0>)
flu
tensor([[[ 0.3412,  0.5426, -0.7675,  0.9639,  0.0595],
         [ 0.2469,  0.4010, -0.3677,  0.2424, -0.7350],
         [ 0.3566,  0.2030, -1.4922,  0.6969, -0.5716],
         [ 0.6725,  0.2972, -1.1109,  0.5329, -0.5262],
         [ 0.7672,  0.7907, -1.6100,  0.7378, -0.0480]]],
       grad_fn=<ViewBackward0>)
t2dm
tensor([[[ 0.4567,  0.5527, -0.6791,  0.9534,  0.0422],
     

In [184]:
generated_sequence = [list(main_vocab.keys())[list(main_vocab.values()).index(id)] for id in generated_tokens]
print("Generated Sequence:", " ".join(generated_sequence))

Generated Sequence: flu born t2dm born flu t2dm t2dm


In [185]:
list(main_vocab.keys())[next_token_id]

't2dm'

In [186]:
generated_tokens

[0, 3, 1, 3, 0, 1, 1]